# AMEX Enterprise Credit Risk Platform
## Notebook 48 -- Roll-Rate Modeling: Validation & Deployment
### Phase 3 . Problem Statement 8: Roll-Rate Modeling

CRISP-DM stage: **Validation & Deployment**. Depends on Notebook 46's real `roll_rate_policy.json` and
Notebook 47's real `roll_rate_modeling_results.json`.

**What this notebook does (real, computed on your machine when you run it):**
- Independently rebuilds Notebook 47's ENTIRE pipeline from scratch (re-fits the per-statement severity
  score on TRAIN, re-derives tertile cutpoints, re-assigns HOLDOUT states, rebuilds the empirical
  transition matrix) and checks every deterministic quantity against Notebook 47's real persisted numbers
  -- this pipeline has zero randomness, so any mismatch would mean a real bug, not sampling variation;
  an immediate `RuntimeError` halts the notebook if reproduction fails
- Bootstraps real 95% confidence intervals (2,000 resamples, `np.random.default_rng` seeded) on this
  problem's two real hard-gate KPIs: the monotonicity (severe/low default-rate) ratio and the transition
  coherence gap, plus the escalation-magnitude ROC-AUC/PR-AUC as secondary statistics
- Runs a real split-half population-stability (PSI) check on the per-statement severity score
- Assembles the full statistical validation summary table (13 real tests) and makes the final honest
  RECOMMENDED / NOT RECOMMENDED FOR PRODUCTION call, combining both real hard-gate KPIs with every
  statistical validation check
- Persists a deployment policy JSON (feature weights, cutpoints, transition matrix -- there is no trained
  classifier to persist here, this technique is a fitted composite score, not a fitted ML model), generates
  a real, runnable, syntax-checked FastAPI service (`roll_rate_scoring_service.py`), live-tests it end to
  end against a real HOLDOUT customer's actual statement (fetched fresh from the raw CSV by its exact
  physical row position, not a synthetic payload), and benchmarks its real API latency
- Writes a deployment readiness checklist and a Word validation/deployment report

**What this notebook does NOT do:** it does not re-verify Notebook 47's Problem 6 covariate stratification
(already exploratory and non-gating; re-running it here would duplicate a large amount of Problem 6 logic
for no genuine reproducibility gain -- an explicit, documented scope decision, not an oversight) and it does
not deploy an actually-running/hosted service (no container, no cloud deploy) -- same scope boundary every
prior Phase 1/2/3 Validation & Deployment notebook in this platform has used.

Zero-fabrication: every threshold is either reused verbatim from Notebook 46's real policy or computed
live from real data in this notebook -- nothing is guessed. The final recommendation reflects whatever
this run's real, measured KPI and statistical-validation results actually are, honestly, even if that
result is NOT RECOMMENDED FOR PRODUCTION.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01-05, 46, 47
# =============================================================================
import os
import sys
import json
import time
import warnings
import importlib.util
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05, 46, 47")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB46_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_46_summary.json"
NB47_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_47_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB46_SUMMARY_PATH, "run 46_roll_rate_modeling_business_understanding.ipynb first"),
    (NB47_SUMMARY_PATH, "run 47_roll_rate_modeling_modeling.ipynb first"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB46_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB46_SUMMARY = json.load(f)
with open(NB47_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB47_SUMMARY = json.load(f)

RR_POLICY_PATH = Path(NB46_SUMMARY["policy_path"])
if not RR_POLICY_PATH.exists():
    raise FileNotFoundError(f"{RR_POLICY_PATH} not found.\nFix: re-run Notebook 46.")
with open(RR_POLICY_PATH, "r", encoding="utf-8") as f:
    RR_POLICY = json.load(f)

STATE_NAMES = RR_POLICY["state_names"]
N_STATES = RR_POLICY["n_states"]
STATE_CUT_PERCENTILES = RR_POLICY["state_cut_percentiles"]
MIN_STATEMENTS_FOR_TRANSITION = RR_POLICY["min_statements_for_transition"]
MONITORED_COLS = sorted(RR_POLICY["monitored_features"]["features"])
RR_KPI_TARGETS = RR_POLICY["kpi_targets"]
P6_COVARIATE = RR_POLICY["problem_6_covariate"]
P6_MODEL_PATH = Path(P6_COVARIATE["model_path"])
P6_PREPROCESSING_PATH = Path(P6_COVARIATE["preprocessing_path"])
P6_WINNING_W = P6_COVARIATE["winning_w"]
P6_RECOMMENDED_FOR_PRODUCTION = P6_COVARIATE["recommended_for_production"]

# --- Notebook 47's own persisted, real results -- this is what Section 4
#     below independently re-derives from scratch and checks against, byte
#     for byte on every deterministic quantity (this pipeline has zero
#     randomness -- no model fit, only correlation-weighted composite
#     z-scores and empirical transition counts -- so any mismatch here means
#     a real bug, not sampling variation; see Notebook 44's identical
#     immediate-raise reasoning). ---
NB47_MODELING_RESULTS_PATH = Path(NB47_SUMMARY["modeling_results_path"])
if not NB47_MODELING_RESULTS_PATH.exists():
    raise FileNotFoundError(f"{NB47_MODELING_RESULTS_PATH} not found.\nFix: re-run Notebook 47.")
with open(NB47_MODELING_RESULTS_PATH, "r", encoding="utf-8") as f:
    NB47_RESULTS = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    """Same 3(+1)-candidate resolver every notebook in this platform uses (see
    Notebook 39 Section 3 / Notebook 47 Section 1 for the full history of
    why): the real known current nested Phase1_Foundation/Problem1.../
    <legacy_folder_name> path is checked FIRST (never trust a summary JSON's
    stored absolute path alone -- Problem 1's own folder was reorganized into
    Phase1_Foundation/ at some point and notebook_02_summary.json's stored
    output_files paths were never rewritten to match), then PILLAR_DIRS, then
    the legacy root-level path, then whatever the summary JSON literally
    recorded (lowest priority, since that's the one most likely to be
    stale)."""
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + f"\n\nnotebook_02_summary.json['output_files'] keys: "
        f"{sorted(NB02_SUMMARY.get('output_files', {}).keys())}\n"
        "Fix: run the notebook that produces this file again, or tell me the real path."
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)

DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

if "roll_rate_deployment" in PILLAR_DIRS:
    RR_DEPLOYMENT_DIR = PILLAR_DIRS["roll_rate_deployment"]
else:
    RR_DEPLOYMENT_DIR = (
        PROJECT_ROOT / "Phase3_Behavioral_Intelligence"
        / "Problem8_Roll_Rate_Modeling" / "deployment"
    )
    print(f"NOTE: 'roll_rate_deployment' not in pillar_dirs -- using fallback: {RR_DEPLOYMENT_DIR}")
RR_DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)
API_SUBDIR = RR_DEPLOYMENT_DIR / "api"
API_SUBDIR.mkdir(parents=True, exist_ok=True)
POLICY_SUBDIR = RR_DEPLOYMENT_DIR / "policy_artifacts"
POLICY_SUBDIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR = RR_DEPLOYMENT_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded Notebook 46 policy from : {RR_POLICY_PATH}")
print(f"Loaded Notebook 47 results from: {NB47_MODELING_RESULTS_PATH}")
print(f"Resolved train_split.csv       : {TRAIN_SPLIT_PATH}")
print(f"Resolved test_split.csv        : {TEST_SPLIT_PATH}")
print(f"STATE_NAMES                    : {STATE_NAMES}")
print(f"Notebook 47 reported: monotonicity KPI={NB47_RESULTS['meets_monotonicity_kpi']}, "
      f"coherence KPI={NB47_RESULTS['meets_coherence_kpi']}")
print(f"Deployment artifacts will be written under: {RR_DEPLOYMENT_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, accuracy_score, precision_score,
        recall_score, f1_score, matthews_corrcoef, confusion_matrix,
    )
except ImportError:
    missing.append("scikit-learn")
try:
    from fastapi.testclient import TestClient
except ImportError:
    missing.append("fastapi")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
    from docx.shared import Inches
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    missing.append("python-docx")
try:
    import importlib.metadata as importlib_metadata
except ImportError:
    import importlib_metadata

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS & LOAD REAL TRAIN/HOLDOUT SPLITS + TARGET
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths & Load Real Train/Holdout Splits + Target")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

TARGET_DF = pl.read_csv(RAW_TRAIN_LABELS_PATH, schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
TRAIN_IDS_DF = pl.read_csv(TRAIN_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8})
HOLDOUT_IDS_DF = pl.read_csv(TEST_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8})
N_TRAIN_CUSTOMERS = TRAIN_IDS_DF.height
N_HOLDOUT_CUSTOMERS = HOLDOUT_IDS_DF.height

print(f"Raw train_data.csv         : {RAW_TRAIN_DATA_PATH}")
print(f"Raw train_labels.csv       : {RAW_TRAIN_LABELS_PATH}")
print(f"Real TRAIN customers (Notebook 02's real split, reused)  : {N_TRAIN_CUSTOMERS:,}")
print(f"Real HOLDOUT customers (Notebook 02's real split, reused): {N_HOLDOUT_CUSTOMERS:,}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: REBUILD & REPRODUCE NOTEBOOK 47'S ENTIRE PIPELINE FROM SCRATCH
#            (INTEGRITY CHECK -- ZERO RANDOMNESS INVOLVED)
# =============================================================================
_section("SECTION 4: Rebuild & Reproduce Notebook 47's Entire Pipeline (Integrity Check)")

# --- This re-declares Notebook 47's Sections 4-9 logic verbatim (not
#     imported -- this platform's notebooks are not yet wired to a shared
#     module, see root ROADMAP.md) so this notebook's reproduction of
#     Notebook 47's numbers is a genuine reproducibility check, not a
#     re-implementation that could silently diverge. This computation has NO
#     randomness (correlation-weighted composite z-scores, tertile
#     quantiles, and empirical transition counts, nothing stochastic), so
#     ANY mismatch below indicates a real bug, not sampling variation --
#     Notebook 44's stricter, immediate-raise pattern is followed rather
#     than Notebook 40's soft-print-then-defer pattern, which exists there
#     only because that notebook re-fits a (seeded, but in-principle
#     stochastic) XGBoost model. ---
_schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
for _c in MONITORED_COLS:
    _schema_overrides[_c] = pl.Float32

_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
    for c in MONITORED_COLS
]

# _csv_row_order: same real determinism fix Notebook 47 found and applied
# (475 duplicate (customer_ID, S_2) statement-date pairs in the raw CSV with
# no sort tiebreaker caused non-reproducible results under Polars' threaded
# streaming engine) -- reproduced here identically so this notebook's own
# reproduction is itself deterministic.
_base_lf = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH, schema_overrides=_schema_overrides)
    .with_row_index("_csv_row_order")
    .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
    .with_columns(_inf_clean_exprs)
    .join(TARGET_DF.lazy(), on="customer_ID", how="inner")
)

_fit_agg_exprs = []
for _c in MONITORED_COLS:
    _fit_agg_exprs += [
        pl.col(_c).mean().alias(f"_mean_{_c}"),
        pl.col(_c).std(ddof=1).alias(f"_std_{_c}"),
        pl.corr(pl.col(_c), pl.col("target")).alias(f"_corr_{_c}"),
    ]

_t0 = time.time()
_fit_row = (
    _base_lf.join(TRAIN_IDS_DF.lazy(), on="customer_ID", how="inner")
    .select(_fit_agg_exprs)
    .collect(engine="streaming")
    .row(0, named=True)
)
print(f"Re-fit weights in {time.time() - _t0:.1f}s. Process RSS: {_rss_gb():.2f} GB")

FEATURE_MEAN, FEATURE_STD, FEATURE_WEIGHT, FEATURE_DIRECTION = {}, {}, {}, {}
for _c in MONITORED_COLS:
    _mean = _fit_row[f"_mean_{_c}"]
    _std = _fit_row[f"_std_{_c}"]
    _corr = _fit_row[f"_corr_{_c}"]
    FEATURE_MEAN[_c] = float(_mean) if _mean is not None else 0.0
    FEATURE_STD[_c] = float(_std) if (_std is not None and _std > 0) else 0.0
    if _corr is None or FEATURE_STD[_c] == 0.0:
        FEATURE_WEIGHT[_c] = 0.0
        FEATURE_DIRECTION[_c] = 0.0
    else:
        FEATURE_WEIGHT[_c] = float(abs(_corr))
        FEATURE_DIRECTION[_c] = 1.0 if _corr >= 0 else -1.0

_wz_cols = []
for _c in MONITORED_COLS:
    _mean, _std, _w, _d = FEATURE_MEAN[_c], FEATURE_STD[_c], FEATURE_WEIGHT[_c], FEATURE_DIRECTION[_c]
    if _std > 0 and _w > 0:
        _expr = ((pl.col(_c) - _mean) / _std * _w * _d).fill_null(0.0).alias(f"_wz_{_c}")
    else:
        _expr = pl.lit(0.0).alias(f"_wz_{_c}")
    _wz_cols.append(_expr)

_scored_lf = (
    _base_lf
    .with_columns(_wz_cols)
    .with_columns(pl.sum_horizontal([f"_wz_{c}" for c in MONITORED_COLS]).alias("SEVERITY_SCORE"))
    .select(["customer_ID", "S_2", "_csv_row_order", "target", "SEVERITY_SCORE"])
)

_train_scores = (
    _scored_lf.join(TRAIN_IDS_DF.lazy(), on="customer_ID", how="inner")
    .select("SEVERITY_SCORE")
    .collect(engine="streaming")["SEVERITY_SCORE"]
)
CUT_LOW = float(_train_scores.quantile(STATE_CUT_PERCENTILES[0] / 100.0, interpolation="linear"))
CUT_HIGH = float(_train_scores.quantile(STATE_CUT_PERCENTILES[1] / 100.0, interpolation="linear"))

_state_expr = (
    pl.when(pl.col("SEVERITY_SCORE") <= CUT_LOW).then(pl.lit(STATE_NAMES[0]))
    .when(pl.col("SEVERITY_SCORE") <= CUT_HIGH).then(pl.lit(STATE_NAMES[1]))
    .otherwise(pl.lit(STATE_NAMES[2]))
    .alias("STATE")
)

HOLDOUT_STATEMENTS = (
    _scored_lf.join(HOLDOUT_IDS_DF.lazy(), on="customer_ID", how="inner")
    .with_columns(_state_expr)
    .sort(["customer_ID", "S_2", "_csv_row_order"])
    .collect(engine="streaming")
)
HOLDOUT_STATEMENTS = HOLDOUT_STATEMENTS.with_columns([
    pl.len().over("customer_ID").alias("_n_statements"),
    pl.int_range(pl.len()).over("customer_ID").alias("_row_idx"),
    pl.col("STATE").shift(1).over("customer_ID").alias("_prev_state"),
])
print(f"Re-scored {HOLDOUT_STATEMENTS.height:,} real HOLDOUT statements. Process RSS: {_rss_gb():.2f} GB")

LATEST_STATE_DF = HOLDOUT_STATEMENTS.filter(pl.col("_row_idx") == pl.col("_n_statements") - 1).select(
    ["customer_ID", "STATE", "target"]
)
_n_latest = LATEST_STATE_DF.height
STATE_DEFAULT_STATS = {}
for _s in STATE_NAMES:
    _sub = LATEST_STATE_DF.filter(pl.col("STATE") == _s)
    _n = _sub.height
    _n_def = int(_sub["target"].sum()) if _n else 0
    STATE_DEFAULT_STATS[_s] = {
        "n": _n, "n_defaulters": _n_def,
        "default_rate": (_n_def / _n) if _n else 0.0,
        "population_pct": (100.0 * _n / _n_latest) if _n_latest else 0.0,
    }
_rates = [STATE_DEFAULT_STATS[_s]["default_rate"] for _s in STATE_NAMES]
MONOTONIC = all(_rates[i] < _rates[i + 1] for i in range(len(_rates) - 1))
_low_rate, _severe_rate = STATE_DEFAULT_STATS[STATE_NAMES[0]]["default_rate"], STATE_DEFAULT_STATS[STATE_NAMES[-1]]["default_rate"]
SEVERE_TO_LOW_RATIO = (_severe_rate / _low_rate) if _low_rate > 0 else float("inf") if _severe_rate > 0 else 0.0
MIN_POPULATION_PCT_ACHIEVED = min(STATE_DEFAULT_STATS[_s]["population_pct"] for _s in STATE_NAMES)
MEETS_MONOTONICITY_KPI = (
    MONOTONIC
    and SEVERE_TO_LOW_RATIO >= RR_KPI_TARGETS["min_default_rate_ratio_top_to_bottom_tier"]
    and MIN_POPULATION_PCT_ACHIEVED >= RR_KPI_TARGETS["min_tier_population_pct"]
)

TRANSITION_PAIRS = HOLDOUT_STATEMENTS.filter(pl.col("_prev_state").is_not_null()).with_columns(
    (pl.col("_row_idx") == (pl.col("_n_statements") - 1)).alias("_is_last_pair")
)
N_TRANSITION_PAIRS = TRANSITION_PAIRS.height
N_TRANSITION_ELIGIBLE_CUSTOMERS = TRANSITION_PAIRS["customer_ID"].n_unique()
_pair_counts = TRANSITION_PAIRS.group_by(["_prev_state", "STATE"]).agg(pl.len().alias("n")).to_dicts()
_count_lookup = {(r["_prev_state"], r["STATE"]): r["n"] for r in _pair_counts}
TRANSITION_MATRIX, TRANSITION_MATRIX_COUNTS = {}, {}
for _i in STATE_NAMES:
    _row_total = sum(_count_lookup.get((_i, _j), 0) for _j in STATE_NAMES)
    TRANSITION_MATRIX[_i] = {_j: (_count_lookup.get((_i, _j), 0) / _row_total if _row_total else 0.0) for _j in STATE_NAMES}
    TRANSITION_MATRIX_COUNTS[_i] = {_j: _count_lookup.get((_i, _j), 0) for _j in STATE_NAMES}
P_SEVERE_SEVERE = TRANSITION_MATRIX[STATE_NAMES[-1]][STATE_NAMES[-1]]
P_LOW_SEVERE = TRANSITION_MATRIX[STATE_NAMES[0]][STATE_NAMES[-1]]
MEETS_COHERENCE_KPI = P_SEVERE_SEVERE > P_LOW_SEVERE

_ordinal = {s: i for i, s in enumerate(STATE_NAMES)}
LAST_TRANSITIONS = TRANSITION_PAIRS.filter(pl.col("_is_last_pair")).with_columns([
    pl.col("_prev_state").replace_strict(_ordinal, return_dtype=pl.Int8).alias("_prev_ord"),
    pl.col("STATE").replace_strict(_ordinal, return_dtype=pl.Int8).alias("_curr_ord"),
]).with_columns([
    (pl.col("_curr_ord") - pl.col("_prev_ord")).alias("ESCALATION_MAGNITUDE"),
    (pl.col("_curr_ord") > pl.col("_prev_ord")).alias("ESCALATED"),
])
N_LAST_TRANSITIONS = LAST_TRANSITIONS.height
_y_true = LAST_TRANSITIONS["target"].to_numpy()
_y_pred_escalated = LAST_TRANSITIONS["ESCALATED"].to_numpy().astype(int)
_escalation_magnitude = LAST_TRANSITIONS["ESCALATION_MAGNITUDE"].to_numpy().astype(float)
_default_rate_escalated = float(LAST_TRANSITIONS.filter(pl.col("ESCALATED"))["target"].mean() or 0.0)
_n_escalated = int(_y_pred_escalated.sum())
_default_rate_not_escalated = float(LAST_TRANSITIONS.filter(~pl.col("ESCALATED"))["target"].mean() or 0.0)
_n_not_escalated = N_LAST_TRANSITIONS - _n_escalated
_cm = confusion_matrix(_y_true, _y_pred_escalated, labels=[0, 1])
_tn, _fp, _fn, _tp = int(_cm[0, 0]), int(_cm[0, 1]), int(_cm[1, 0]), int(_cm[1, 1])
ESCALATION_METRICS_SUITE = {
    "n_last_transitions": N_LAST_TRANSITIONS, "n_escalated": _n_escalated, "n_not_escalated": _n_not_escalated,
    "default_rate_escalated": _default_rate_escalated, "default_rate_not_escalated": _default_rate_not_escalated,
    "accuracy": float(accuracy_score(_y_true, _y_pred_escalated)),
    "precision": float(precision_score(_y_true, _y_pred_escalated, zero_division=0)),
    "recall": float(recall_score(_y_true, _y_pred_escalated, zero_division=0)),
    "f1": float(f1_score(_y_true, _y_pred_escalated, zero_division=0)),
    "specificity": (_tn / (_tn + _fp)) if (_tn + _fp) else 0.0,
    "mcc": float(matthews_corrcoef(_y_true, _y_pred_escalated)) if len(set(_y_pred_escalated)) > 1 else 0.0,
    "confusion_matrix": {"tn": _tn, "fp": _fp, "fn": _fn, "tp": _tp},
    "roc_auc_magnitude": float(roc_auc_score(_y_true, _escalation_magnitude)) if len(set(_y_true)) > 1 else None,
    "pr_auc_magnitude": float(average_precision_score(_y_true, _escalation_magnitude)) if len(set(_y_true)) > 1 else None,
}

# --- The reproduction gate: compare EVERY deterministic quantity that feeds
#     a downstream hard-gate KPI or bootstrap CI against Notebook 47's real
#     persisted numbers. Exact equality for counts/booleans; a tight 1e-9
#     tolerance for floats (not 1e-6 -- unlike Notebook 40's stochastic
#     model-fit comparison, this pipeline has genuinely zero randomness, so
#     the numbers should match to floating-point precision, not just
#     "close"). ---
_reproduction_checks = {
    "cut_low": (abs(CUT_LOW - NB47_RESULTS["cut_low"]) < 1e-9, CUT_LOW, NB47_RESULTS["cut_low"]),
    "cut_high": (abs(CUT_HIGH - NB47_RESULTS["cut_high"]) < 1e-9, CUT_HIGH, NB47_RESULTS["cut_high"]),
    "monotonic": (MONOTONIC == NB47_RESULTS["monotonic"], MONOTONIC, NB47_RESULTS["monotonic"]),
    "severe_to_low_default_rate_ratio": (
        abs(SEVERE_TO_LOW_RATIO - NB47_RESULTS["severe_to_low_default_rate_ratio"]) < 1e-9,
        SEVERE_TO_LOW_RATIO, NB47_RESULTS["severe_to_low_default_rate_ratio"],
    ),
    "meets_monotonicity_kpi": (
        MEETS_MONOTONICITY_KPI == NB47_RESULTS["meets_monotonicity_kpi"],
        MEETS_MONOTONICITY_KPI, NB47_RESULTS["meets_monotonicity_kpi"],
    ),
    "n_transition_pairs": (N_TRANSITION_PAIRS == NB47_RESULTS["n_transition_pairs"], N_TRANSITION_PAIRS, NB47_RESULTS["n_transition_pairs"]),
    "p_severe_severe": (abs(P_SEVERE_SEVERE - NB47_RESULTS["p_severe_severe"]) < 1e-9, P_SEVERE_SEVERE, NB47_RESULTS["p_severe_severe"]),
    "p_low_severe": (abs(P_LOW_SEVERE - NB47_RESULTS["p_low_severe"]) < 1e-9, P_LOW_SEVERE, NB47_RESULTS["p_low_severe"]),
    "meets_coherence_kpi": (
        MEETS_COHERENCE_KPI == NB47_RESULTS["meets_coherence_kpi"],
        MEETS_COHERENCE_KPI, NB47_RESULTS["meets_coherence_kpi"],
    ),
    "escalation_default_rate_escalated": (
        abs(_default_rate_escalated - NB47_RESULTS["escalation_metrics_suite"]["default_rate_escalated"]) < 1e-9,
        _default_rate_escalated, NB47_RESULTS["escalation_metrics_suite"]["default_rate_escalated"],
    ),
}
print("Reproduction check (Notebook 48 fresh computation vs. Notebook 47's real persisted numbers):")
_reproduction_matches = True
for _k, (_ok, _mine, _theirs) in _reproduction_checks.items():
    print(f"  [{'MATCH' if _ok else 'MISMATCH'}] {_k}: reproduced={_mine!r}  Notebook 47 reported={_theirs!r}")
    _reproduction_matches = _reproduction_matches and _ok
if not _reproduction_matches:
    raise RuntimeError(
        "Notebook 48's reproduction does NOT match Notebook 47's persisted numbers -- investigate "
        "before proceeding (this pipeline has no randomness, so any mismatch indicates a real bug, "
        "not sampling variation)."
    )
print(f"\nReproduction matches Notebook 47 exactly (zero-randomness pipeline): {_reproduction_matches}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: BOOTSTRAP CONFIDENCE INTERVALS -- MONOTONICITY RATIO (PRIMARY),
#            COHERENCE GAP (PRIMARY), ESCALATION-MAGNITUDE AUC/PR-AUC (SECONDARY)
# =============================================================================
_section("SECTION 5: Bootstrap Confidence Intervals")

# --- Same plain-numpy empirical-percentile bootstrap every prior Phase-3
#     notebook uses (Notebook 40 Section 7, Notebook 44 Section 9) -- no
#     external CI library, N_BOOTSTRAP=2000, np.random.default_rng(RANDOM_SEED),
#     draw-with-replacement over the real holdout population, 95% CI via
#     np.percentile([2.5, 97.5]). Three statistics get a CI here, matching
#     this problem's three real hard-gate/reporting quantities: the
#     monotonicity ratio (this problem's primary domain KPI, analogous to
#     Notebook 44's "lift"), the coherence gap (this problem's second hard
#     gate, with no prior-problem precedent), and the escalation-magnitude
#     AUC/PR-AUC (secondary, analogous to Notebook 40's AUC/PR-AUC CIs). ---
N_BOOTSTRAP = 2000
_rng = np.random.default_rng(RANDOM_SEED)

_latest_states = LATEST_STATE_DF["STATE"].to_numpy()
_latest_targets = LATEST_STATE_DF["target"].to_numpy()
_n_latest_boot = len(_latest_states)

_prev_states_arr = TRANSITION_PAIRS["_prev_state"].to_numpy()
_curr_states_arr = TRANSITION_PAIRS["STATE"].to_numpy()
_n_pairs_boot = len(_prev_states_arr)

_boot_ratios = np.full(N_BOOTSTRAP, np.nan)
_boot_coherence_gaps = np.full(N_BOOTSTRAP, np.nan)
_boot_aucs = np.full(N_BOOTSTRAP, np.nan)
_boot_pr_aucs = np.full(N_BOOTSTRAP, np.nan)

_t0 = time.time()
for _b in range(N_BOOTSTRAP):
    # --- Monotonicity ratio bootstrap (resample latest-state customers) ---
    _idx1 = _rng.integers(0, _n_latest_boot, size=_n_latest_boot)
    _s_boot, _t_boot = _latest_states[_idx1], _latest_targets[_idx1]
    _low_mask = _s_boot == STATE_NAMES[0]
    _severe_mask = _s_boot == STATE_NAMES[-1]
    if _low_mask.sum() > 0 and _severe_mask.sum() > 0:
        _low_rate_boot = float(_t_boot[_low_mask].mean())
        _severe_rate_boot = float(_t_boot[_severe_mask].mean())
        if _low_rate_boot > 0:
            _boot_ratios[_b] = _severe_rate_boot / _low_rate_boot

    # --- Coherence gap bootstrap (resample transition pairs) ---
    _idx2 = _rng.integers(0, _n_pairs_boot, size=_n_pairs_boot)
    _p_boot, _c_boot = _prev_states_arr[_idx2], _curr_states_arr[_idx2]
    _severe_prev_mask = _p_boot == STATE_NAMES[-1]
    _low_prev_mask = _p_boot == STATE_NAMES[0]
    if _severe_prev_mask.sum() > 0 and _low_prev_mask.sum() > 0:
        _p_ss_boot = float((_c_boot[_severe_prev_mask] == STATE_NAMES[-1]).mean())
        _p_ls_boot = float((_c_boot[_low_prev_mask] == STATE_NAMES[-1]).mean())
        _boot_coherence_gaps[_b] = _p_ss_boot - _p_ls_boot

    # --- Escalation-magnitude AUC/PR-AUC bootstrap (resample last transitions) ---
    _idx3 = _rng.integers(0, N_LAST_TRANSITIONS, size=N_LAST_TRANSITIONS)
    _yt_boot, _mag_boot = _y_true[_idx3], _escalation_magnitude[_idx3]
    if len(set(_yt_boot)) > 1:
        _boot_aucs[_b] = roc_auc_score(_yt_boot, _mag_boot)
        _boot_pr_aucs[_b] = average_precision_score(_yt_boot, _mag_boot)

print(f"Ran {N_BOOTSTRAP:,} bootstrap iterations in {time.time() - _t0:.1f}s")

_valid_ratios = _boot_ratios[~np.isnan(_boot_ratios)]
_valid_gaps = _boot_coherence_gaps[~np.isnan(_boot_coherence_gaps)]
_valid_aucs = _boot_aucs[~np.isnan(_boot_aucs)]
_valid_pr_aucs = _boot_pr_aucs[~np.isnan(_boot_pr_aucs)]
print(f"Valid bootstrap draws -- ratio: {len(_valid_ratios):,}, coherence gap: {len(_valid_gaps):,}, "
      f"AUC: {len(_valid_aucs):,}, PR-AUC: {len(_valid_pr_aucs):,} (of {N_BOOTSTRAP:,})")

RATIO_CI_LOWER, RATIO_CI_UPPER = (float(x) for x in np.percentile(_valid_ratios, [2.5, 97.5])) if len(_valid_ratios) else (float("nan"), float("nan"))
COHERENCE_GAP_CI_LOWER, COHERENCE_GAP_CI_UPPER = (float(x) for x in np.percentile(_valid_gaps, [2.5, 97.5])) if len(_valid_gaps) else (float("nan"), float("nan"))
AUC_CI_LOWER, AUC_CI_UPPER = (float(x) for x in np.percentile(_valid_aucs, [2.5, 97.5])) if len(_valid_aucs) else (float("nan"), float("nan"))
PR_AUC_CI_LOWER, PR_AUC_CI_UPPER = (float(x) for x in np.percentile(_valid_pr_aucs, [2.5, 97.5])) if len(_valid_pr_aucs) else (float("nan"), float("nan"))

_no_skill_pr_baseline = float(_y_true.mean()) if len(_y_true) else 0.0
_ratio_ci_excludes_no_effect = RATIO_CI_LOWER > 1.0
_coherence_ci_excludes_zero = COHERENCE_GAP_CI_LOWER > 0.0
_auc_ci_excludes_random = AUC_CI_LOWER > 0.5
_pr_auc_ci_excludes_noskill = PR_AUC_CI_LOWER > _no_skill_pr_baseline

print(f"\nSevere/Low default-rate ratio  -- point estimate: {SEVERE_TO_LOW_RATIO:.3f}x, "
      f"95% CI: [{RATIO_CI_LOWER:.3f}, {RATIO_CI_UPPER:.3f}]  (CI lower > 1.0: {_ratio_ci_excludes_no_effect})")
print(f"Coherence gap P(Sev->Sev)-P(Low->Sev) -- point estimate: {P_SEVERE_SEVERE - P_LOW_SEVERE:.4f}, "
      f"95% CI: [{COHERENCE_GAP_CI_LOWER:.4f}, {COHERENCE_GAP_CI_UPPER:.4f}]  (CI lower > 0.0: {_coherence_ci_excludes_zero})")
print(f"Escalation-magnitude ROC-AUC    -- point estimate: {ESCALATION_METRICS_SUITE['roc_auc_magnitude']:.4f}, "
      f"95% CI: [{AUC_CI_LOWER:.4f}, {AUC_CI_UPPER:.4f}]  (CI lower > 0.5: {_auc_ci_excludes_random})")
print(f"Escalation-magnitude PR-AUC     -- point estimate: {ESCALATION_METRICS_SUITE['pr_auc_magnitude']:.4f}, "
      f"95% CI: [{PR_AUC_CI_LOWER:.4f}, {PR_AUC_CI_UPPER:.4f}]  (CI lower > {_no_skill_pr_baseline:.4f} no-skill: {_pr_auc_ci_excludes_noskill})")
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: SPLIT-HALF POPULATION STABILITY (PSI) ON THE SEVERITY SCORE
# =============================================================================
_section("SECTION 6: Split-Half Population Stability (PSI) on the Severity Score")

# --- Same honest framing this platform already documents for Notebook 21/
#     monitoring_job.py's PSI (reused verbatim by Notebooks 36/40/44): a
#     random split-half stability proxy on the single available holdout
#     population, not a genuine time-based drift measurement -- there is no
#     second real time period to compare against yet. Bin edges come from
#     the TRAIN score distribution's real deciles, applied to two random
#     halves of the HOLDOUT statement-level severity-score population. ---
_holdout_scores = HOLDOUT_STATEMENTS["SEVERITY_SCORE"].to_numpy()
_n_holdout_scores = len(_holdout_scores)
_train_scores_np = _train_scores.to_numpy()
_train_edges = np.quantile(_train_scores_np, np.linspace(0, 1, 11))
_train_edges[0], _train_edges[-1] = -np.inf, np.inf
_perm = _rng.permutation(_n_holdout_scores)
_half = _n_holdout_scores // 2
_half_a = _holdout_scores[_perm[:_half]]
_half_b = _holdout_scores[_perm[_half:]]
_share_a = np.histogram(_half_a, bins=_train_edges)[0] / len(_half_a)
_share_b = np.histogram(_half_b, bins=_train_edges)[0] / len(_half_b)
_share_a = np.clip(_share_a, 1e-4, None)
_share_b = np.clip(_share_b, 1e-4, None)
SCORE_PSI_SPLIT_HALF = float(((_share_a - _share_b) * np.log(_share_a / _share_b)).sum())
_psi_target = 0.10  # ASSUMPTION: standard industry PSI stability threshold (<0.10 = no significant shift)
print(f"Split-half PSI on real per-statement SEVERITY_SCORE (n={_n_holdout_scores:,}): {SCORE_PSI_SPLIT_HALF:.4f}  "
      f"(target < {_psi_target}, {'PASS' if SCORE_PSI_SPLIT_HALF < _psi_target else 'FAIL'})")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: FULL METRICS-SUITE STATISTICAL VALIDATION SUMMARY TABLE
# =============================================================================
_section("SECTION 7: Full Metrics-Suite Statistical Validation Summary Table")

# --- Per the user's 2026-08-25 standing directive, the FULL classification/
#     validation metrics suite is assembled into one table here -- every row
#     below is a real, measured value from Sections 4-6 above, none
#     fabricated or assumed. MEETS_KPI combines BOTH of Notebook 47's real
#     hard gates (monotonicity AND coherence) -- either failing means the
#     technique has not cleared this problem's bar, exactly mirroring how
#     Notebooks 40/44 combine their own single hard-gate KPI with the
#     statistical validation table's pass/fail. ---
MEETS_KPI = bool(MEETS_MONOTONICITY_KPI and MEETS_COHERENCE_KPI)

statistical_validation_rows = [
    {"test": "Severe/Low default-rate ratio (reproduced)", "value": round(SEVERE_TO_LOW_RATIO, 3),
     "target": f">={RR_KPI_TARGETS['min_default_rate_ratio_top_to_bottom_tier']}x", "pass": bool(MEETS_MONOTONICITY_KPI)},
    {"test": "Severe/Low ratio 95% CI lower bound", "value": round(RATIO_CI_LOWER, 3),
     "target": ">1.0 (no-effect)", "pass": bool(_ratio_ci_excludes_no_effect)},
    {"test": "Monotonic default rate Low < Moderate < Severe (reproduced)", "value": MONOTONIC,
     "target": "True", "pass": bool(MONOTONIC)},
    {"test": "Minimum tier population share (reproduced)", "value": round(MIN_POPULATION_PCT_ACHIEVED, 1),
     "target": f">={RR_KPI_TARGETS['min_tier_population_pct']}%", "pass": bool(MIN_POPULATION_PCT_ACHIEVED >= RR_KPI_TARGETS["min_tier_population_pct"])},
    {"test": "P(Severe -> Severe) [persistence] (reproduced)", "value": round(P_SEVERE_SEVERE, 4),
     "target": "reported", "pass": True},
    {"test": "P(Low -> Severe) [single-step jump] (reproduced)", "value": round(P_LOW_SEVERE, 4),
     "target": "reported", "pass": True},
    {"test": "Coherence gap P(Sev->Sev) - P(Low->Sev) (reproduced)", "value": round(P_SEVERE_SEVERE - P_LOW_SEVERE, 4),
     "target": ">0.0", "pass": bool(MEETS_COHERENCE_KPI)},
    {"test": "Coherence gap 95% CI lower bound", "value": round(COHERENCE_GAP_CI_LOWER, 4),
     "target": ">0.0 (no-effect)", "pass": bool(_coherence_ci_excludes_zero)},
    {"test": "Escalation-magnitude ROC-AUC (reproduced)", "value": round(ESCALATION_METRICS_SUITE["roc_auc_magnitude"], 4) if ESCALATION_METRICS_SUITE["roc_auc_magnitude"] is not None else None,
     "target": ">0.5", "pass": bool(ESCALATION_METRICS_SUITE["roc_auc_magnitude"] is not None and ESCALATION_METRICS_SUITE["roc_auc_magnitude"] > 0.5)},
    {"test": "Escalation-magnitude ROC-AUC 95% CI lower bound", "value": round(AUC_CI_LOWER, 4),
     "target": ">0.5 (random)", "pass": bool(_auc_ci_excludes_random)},
    {"test": "Escalation-magnitude PR-AUC (reproduced)", "value": round(ESCALATION_METRICS_SUITE["pr_auc_magnitude"], 4) if ESCALATION_METRICS_SUITE["pr_auc_magnitude"] is not None else None,
     "target": f">{_no_skill_pr_baseline:.4f} (no-skill)", "pass": bool(ESCALATION_METRICS_SUITE["pr_auc_magnitude"] is not None and ESCALATION_METRICS_SUITE["pr_auc_magnitude"] > _no_skill_pr_baseline)},
    {"test": "Escalation-magnitude PR-AUC 95% CI lower bound", "value": round(PR_AUC_CI_LOWER, 4),
     "target": f">{_no_skill_pr_baseline:.4f} (no-skill)", "pass": bool(_pr_auc_ci_excludes_noskill)},
    {"test": "Split-half severity-score PSI", "value": round(SCORE_PSI_SPLIT_HALF, 4),
     "target": f"<{_psi_target}", "pass": bool(SCORE_PSI_SPLIT_HALF < _psi_target)},
]
statistical_validation_df = pd.DataFrame(statistical_validation_rows)
statistical_validation_path = RR_DEPLOYMENT_DIR / "roll_rate_statistical_validation.csv"
statistical_validation_df.to_csv(statistical_validation_path, index=False)
print(statistical_validation_df.to_string(index=False))
ALL_STAT_CHECKS_PASS = bool(statistical_validation_df["pass"].all())
print(f"\nMEETS_KPI (monotonicity AND coherence, both hard gates): {MEETS_KPI}")
print(f"All statistical checks pass: {ALL_STAT_CHECKS_PASS}")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: HONEST LIMITATION -- DEPLOYMENT SCOPE & ASSUMPTIONS
# =============================================================================
_section("SECTION 8: Honest Limitation -- Deployment Scope & Assumptions")

DEPLOYMENT_LIMITATION = {
    "technique_scope": (
        "This is a Markov transition-probability matrix over a per-STATEMENT severity score, NOT a "
        "trained classifier -- there is no model to overfit or drift in the usual sense; the deployable "
        "artifact is a fitted set of composite-score weights, two tertile cutpoints, and an empirical "
        "3x3 transition matrix, all frozen at TRAIN-fit time."
    ),
    "kpi_status": (
        f"Meets BOTH real hard gates (monotonicity ratio {SEVERE_TO_LOW_RATIO:.2f}x >= "
        f"{RR_KPI_TARGETS['min_default_rate_ratio_top_to_bottom_tier']}x, and coherence "
        f"P(Severe->Severe)={P_SEVERE_SEVERE:.3f} > P(Low->Severe)={P_LOW_SEVERE:.3f})."
        if MEETS_KPI else
        f"Does NOT meet at least one real hard gate on this run (monotonicity: {MEETS_MONOTONICITY_KPI}, "
        f"coherence: {MEETS_COHERENCE_KPI}) -- see Section 4/7 above for the exact measured values. "
        "This notebook proceeds to package the technique for completeness and transparency, but the "
        "final recommendation below reflects this honestly."
    ),
    "coherence_kpi_caveat": (
        "A failed coherence check specifically would most plausibly reflect the lack of genuine "
        "temporal autocorrelation in a small/synthetic test fixture (statements generated independently "
        "per month) rather than a flaw in the technique itself -- real delinquency behavior is expected "
        "to show much stronger month-to-month persistence. This is a plausible explanation, not a "
        "verified one; the real Kaggle data result may differ and should be re-checked when this "
        "notebook is run against it."
        if not MEETS_COHERENCE_KPI else
        "The coherence gate passed on this run -- persistence in the Severe state genuinely exceeds the "
        "single-step jump probability from the Low state."
    ),
    "feature_and_label_provenance": (
        "Monitored feature universe reused from Problem 4's real, already-vetted severity-scoring bundle "
        "(Notebook 28) -- the FEATURE UNIVERSE only, not Problem 4's customer-level fitted weights, which "
        "would not match this problem's statement-level population (documented in Notebook 46). Composite "
        "score weights/cutpoints are fit fresh on TRAIN statements in this pipeline's own Section 4."
    ),
    "problem_6_stratification_caveat": (
        f"Notebook 47's Problem 6 covariate stratification is EXPLORATORY ONLY and is not independently "
        f"re-verified in this notebook (its own build already duplicates a large amount of Problem 6 "
        f"logic; re-running it again here would not add a genuine reproducibility signal beyond what "
        f"Notebook 47 already established). Problem 6's own honest recommendation status "
        f"(recommended_for_production={P6_RECOMMENDED_FOR_PRODUCTION}) is carried through unchanged -- "
        f"this stratification result should not be read as validating Problem 6's model."
    ),
    "bootstrap_caveat": (
        "The coherence-gap and ratio bootstrap CIs resample transition pairs / latest-state customers "
        "independently, without block-resampling by customer -- a customer contributing multiple "
        "transition pairs is not kept together across a resample. This is the same simple i.i.d. "
        "bootstrap convention every prior notebook in this platform uses (no block-bootstrap-by-customer "
        "precedent exists yet); it understates the true CI width to the extent transition pairs from the "
        "same customer are correlated, a known simplification, not a fabricated fix."
    ),
}
for _k, _v in DEPLOYMENT_LIMITATION.items():
    print(f"{_k}:\n  {_v}\n")
print("✅ Section 8 complete.")


# =============================================================================
# SECTION 9: PERSIST ROLL-RATE DEPLOYMENT POLICY & VALIDATION ARTIFACTS
# =============================================================================
_section("SECTION 9: Persist Roll-Rate Deployment Policy & Validation Artifacts")

# --- There is no trained classifier to persist (this technique is a fitted
#     composite-score + transition matrix, not a fitted ML model) -- what
#     gets persisted is the DEPLOYMENT POLICY: the exact feature weights/
#     means/stds/directions, cutpoints, and empirical transition matrix a
#     real-time service needs to reproduce this notebook's state-assignment
#     and transition-lookup computation on a NEW customer's statement at
#     inference time. Same substitution Notebook 44 made for its own
#     rule-based technique (a policy JSON in place of a joblib model). ---
ROLL_RATE_DEPLOYMENT_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "state_names": STATE_NAMES,
    "cut_low": CUT_LOW,
    "cut_high": CUT_HIGH,
    "feature_weights": {"weights": FEATURE_WEIGHT, "directions": FEATURE_DIRECTION,
                         "means": FEATURE_MEAN, "stds": FEATURE_STD},
    "monitored_features": MONITORED_COLS,
    "transition_matrix": TRANSITION_MATRIX,
    "transition_matrix_counts": TRANSITION_MATRIX_COUNTS,
    "min_statements_for_transition": MIN_STATEMENTS_FOR_TRANSITION,
    "meets_monotonicity_kpi": MEETS_MONOTONICITY_KPI,
    "meets_coherence_kpi": MEETS_COHERENCE_KPI,
    "meets_kpi_target": MEETS_KPI,
    "recommended_for_production": bool(MEETS_KPI and ALL_STAT_CHECKS_PASS),
    "random_seed": RANDOM_SEED,
}
deployment_policy_path = POLICY_SUBDIR / "roll_rate_deployment_policy.json"
with open(deployment_policy_path, "w", encoding="utf-8") as f:
    json.dump(ROLL_RATE_DEPLOYMENT_POLICY, f, indent=2)
print(f"✅ Saved -> {deployment_policy_path} ({deployment_policy_path.stat().st_size / 1e3:.1f} KB)")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: GENERATE roll_rate_scoring_service.py -- REAL, RUNNABLE FASTAPI SERVICE
# =============================================================================
_section("SECTION 10: Generate roll_rate_scoring_service.py -- Real FastAPI Service")

# --- Same plain string-list generation pattern Notebooks 10/22/36/40/44
#     established (avoids f-string brace-escaping on the generated source's
#     own literal braces), same policy-JSON-driven, env-var-overridable
#     config convention Notebook 44 established for a rule-based (non-
#     classifier) technique. Genuinely different shape from every prior
#     service in this platform: it takes a customer's CURRENT statement's
#     raw monitored features (to compute a severity score and assign a
#     state) and an OPTIONAL previous state (to look up the empirical
#     next-state transition-probability distribution from that state). ---
_deployment_policy_path_str = str(deployment_policy_path)

ROLL_RATE_SERVICE_TEMPLATE = "\n".join([
    "# AMEX Enterprise Credit Risk Platform -- Roll-Rate Modeling Scoring API.",
    "# Auto-generated by 48_roll_rate_modeling_validation_deployment.ipynb.",
    "# Scores a customer's CURRENT statement into a severity state (Low/Moderate/Severe) and, if the",
    "# customer's PREVIOUS state is also supplied, returns the empirical next-state transition",
    "# probabilities looked up from the real Notebook 47 transition matrix.",
    "# Run with:",
    "#     uvicorn roll_rate_scoring_service:app --host 0.0.0.0 --port 8005",
    "import json",
    "import os",
    "from pathlib import Path",
    "from typing import Dict, Optional",
    "",
    "from fastapi import FastAPI, HTTPException",
    "from pydantic import BaseModel, create_model",
    "",
    "POLICY_PATH = Path(os.environ.get(\"AMEX_RR_POLICY_PATH\", r\"__POLICY_PATH_TOKEN__\"))",
    "with open(POLICY_PATH, \"r\", encoding=\"utf-8\") as _f:",
    "    _POLICY = json.load(_f)",
    "",
    "STATE_NAMES = _POLICY[\"state_names\"]",
    "CUT_LOW = _POLICY[\"cut_low\"]",
    "CUT_HIGH = _POLICY[\"cut_high\"]",
    "_WEIGHTS = _POLICY[\"feature_weights\"][\"weights\"]",
    "_DIRECTIONS = _POLICY[\"feature_weights\"][\"directions\"]",
    "_MEANS = _POLICY[\"feature_weights\"][\"means\"]",
    "_STDS = _POLICY[\"feature_weights\"][\"stds\"]",
    "MONITORED_FEATURES = _POLICY[\"monitored_features\"]",
    "TRANSITION_MATRIX = _POLICY[\"transition_matrix\"]",
    "RECOMMENDED_FOR_PRODUCTION = _POLICY[\"recommended_for_production\"]",
    "_ORDINAL = {s: i for i, s in enumerate(STATE_NAMES)}",
    "",
    "_schema_fields = {_c: (Optional[float], None) for _c in MONITORED_FEATURES}",
    "CurrentStatement = create_model(\"CurrentStatement\", **_schema_fields)",
    "",
    "",
    "class ScoreRequest(BaseModel):",
    "    customer_id: Optional[str] = None",
    "    current_statement: CurrentStatement",
    "    # The customer's state as of their PREVIOUS statement, if known -- e.g. from a prior call to",
    "    # this same endpoint. Omit for a customer's first-ever scored statement.",
    "    previous_state: Optional[str] = None",
    "",
    "",
    "class ScoreResponse(BaseModel):",
    "    customer_id: Optional[str] = None",
    "    severity_score: float",
    "    state: str",
    "    previous_state: Optional[str] = None",
    "    escalated: Optional[bool] = None",
    "    transition_probabilities: Optional[Dict[str, float]] = None",
    f"    meets_kpi_target: bool = {MEETS_KPI}",
    f"    recommended_for_production: bool = {bool(MEETS_KPI and ALL_STAT_CHECKS_PASS)}",
    "",
    "",
    "def compute_severity_score(statement: dict) -> float:",
    "    score = 0.0",
    "    for col in MONITORED_FEATURES:",
    "        val = statement.get(col)",
    "        if val is None:",
    "            continue",
    "        std, w = _STDS[col], _WEIGHTS[col]",
    "        if std > 0 and w > 0:",
    "            score += (val - _MEANS[col]) / std * w * _DIRECTIONS[col]",
    "    return score",
    "",
    "",
    "def assign_state(score: float) -> str:",
    "    if score <= CUT_LOW:",
    "        return STATE_NAMES[0]",
    "    if score <= CUT_HIGH:",
    "        return STATE_NAMES[1]",
    "    return STATE_NAMES[2]",
    "",
    "",
    "app = FastAPI(",
    "    title=\"AMEX Enterprise Credit Risk Platform -- Roll-Rate Modeling Scoring API\",",
    "    description=\"Assigns a customer's current statement a delinquency-severity state and looks up \"",
    "                \"real empirical next-state transition probabilities from a prior state. See \"",
    "                \"/model-info for the real validation metrics behind this technique.\",",
    "    version=\"1.0.0\",",
    ")",
    "",
    "",
    "@app.get(\"/health\")",
    "def health():",
    "    return {\"status\": \"ok\", \"state_names\": STATE_NAMES}",
    "",
    "",
    "@app.get(\"/model-info\")",
    "def model_info():",
    "    return {",
    "        \"state_names\": STATE_NAMES,",
    "        \"cut_low\": CUT_LOW,",
    "        \"cut_high\": CUT_HIGH,",
    f"        \"meets_monotonicity_kpi\": {MEETS_MONOTONICITY_KPI},",
    f"        \"meets_coherence_kpi\": {MEETS_COHERENCE_KPI},",
    f"        \"meets_kpi_target\": {MEETS_KPI},",
    "        \"recommended_for_production\": RECOMMENDED_FOR_PRODUCTION,",
    "        \"transition_matrix\": TRANSITION_MATRIX,",
    "    }",
    "",
    "",
    "@app.post(\"/score\", response_model=ScoreResponse)",
    "def score(request: ScoreRequest):",
    "    statement = request.current_statement.dict() if hasattr(request.current_statement, \"dict\") \\",
    "        else request.current_statement.model_dump()",
    "    try:",
    "        severity_score = compute_severity_score(statement)",
    "        state = assign_state(severity_score)",
    "    except Exception as exc:",
    "        raise HTTPException(status_code=500, detail=\"Scoring failed: \" + str(exc))",
    "",
    "    escalated = None",
    "    transition_probabilities = None",
    "    if request.previous_state is not None:",
    "        if request.previous_state not in STATE_NAMES:",
    "            raise HTTPException(status_code=400, detail=f\"Unknown previous_state: {request.previous_state}\")",
    "        escalated = _ORDINAL[state] > _ORDINAL[request.previous_state]",
    "        transition_probabilities = TRANSITION_MATRIX[request.previous_state]",
    "",
    "    return ScoreResponse(",
    "        customer_id=request.customer_id, severity_score=severity_score, state=state,",
    "        previous_state=request.previous_state, escalated=escalated,",
    "        transition_probabilities=transition_probabilities,",
    "    )",
    "",
])
ROLL_RATE_SERVICE_SOURCE = ROLL_RATE_SERVICE_TEMPLATE.replace("__POLICY_PATH_TOKEN__", _deployment_policy_path_str)

service_py_path = API_SUBDIR / "roll_rate_scoring_service.py"
with open(service_py_path, "w", encoding="utf-8") as f:
    f.write(ROLL_RATE_SERVICE_SOURCE)
compile(ROLL_RATE_SERVICE_SOURCE, str(service_py_path), "exec")
print(f"Generated {len(ROLL_RATE_SERVICE_SOURCE.splitlines())} lines, syntax-checked OK.")
print(f"✅ Saved -> {service_py_path}")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: GENERATE .env.example & requirements-api.txt
# =============================================================================
_section("SECTION 11: Generate .env.example & requirements-api.txt")

ENV_EXAMPLE = f"""# Copy to .env and edit if this machine's policy artifact path differs from the default.
AMEX_RR_POLICY_PATH={deployment_policy_path}
"""
env_example_path = API_SUBDIR / ".env.example"
with open(env_example_path, "w", encoding="utf-8") as f:
    f.write(ENV_EXAMPLE)

_api_packages = ["fastapi", "uvicorn", "pydantic", "numpy"]
_api_pkg_versions = {}
for _pkg in _api_packages:
    try:
        _api_pkg_versions[_pkg] = importlib_metadata.version(_pkg)
    except importlib_metadata.PackageNotFoundError:
        _api_pkg_versions[_pkg] = None

requirements_api_path = API_SUBDIR / "requirements-api.txt"
with open(requirements_api_path, "w", encoding="utf-8") as f:
    f.write(f"# Minimal runtime dependencies for roll_rate_scoring_service.py -- auto-generated "
             f"{datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    for _pkg, _ver in _api_pkg_versions.items():
        f.write(f"{_pkg}=={_ver}\n" if _ver else f"# {_pkg}  -- not installed here\n")

print(f"✅ Saved -> {env_example_path}")
print(f"✅ Saved -> {requirements_api_path}")
print("\n✅ Section 11 complete.")


# =============================================================================
# SECTION 12: LIVE SELF-TEST -- IMPORT THE GENERATED SERVICE & DRIVE IT WITH
#             A REAL HOLDOUT CUSTOMER'S ACTUAL STATEMENT
# =============================================================================
_section("SECTION 12: Live Self-Test -- Import the Generated Service & Drive It")

# --- Imports the EXACT file just written to disk -- proves the delivered
#     artifact works, not just an in-notebook copy of the same logic. Uses a
#     real transition-eligible HOLDOUT customer's ACTUAL last statement's raw
#     feature values (fetched fresh from the raw CSV by its exact
#     _csv_row_order, so it is genuinely the same physical row this
#     notebook's own pipeline scored), not a synthetic payload. ---
os.environ["AMEX_RR_POLICY_PATH"] = str(deployment_policy_path)
_spec = importlib.util.spec_from_file_location("amex_roll_rate_service", str(service_py_path))
_service_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_service_module)
client = TestClient(_service_module.app)

_health_resp = client.get("/health")
assert _health_resp.status_code == 200, f"/health returned {_health_resp.status_code}"
print(f"GET /health     -> {_health_resp.status_code}  {_health_resp.json()}")

_info_resp = client.get("/model-info")
assert _info_resp.status_code == 200, f"/model-info returned {_info_resp.status_code}"
print(f"GET /model-info -> {_info_resp.status_code}  meets_kpi_target={_info_resp.json()['meets_kpi_target']}")

_sample_row = LAST_TRANSITIONS.row(0, named=True)
SAMPLE_CUSTOMER_ID = _sample_row["customer_ID"]
SAMPLE_PREVIOUS_STATE = _sample_row["_prev_state"]
SAMPLE_EXPECTED_STATE = _sample_row["STATE"]
SAMPLE_EXPECTED_SCORE = float(_sample_row["SEVERITY_SCORE"])
SAMPLE_CSV_ROW_ORDER = _sample_row["_csv_row_order"]

_raw_sample = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH, schema_overrides=_schema_overrides)
    .with_row_index("_csv_row_order")
    .filter(pl.col("_csv_row_order") == SAMPLE_CSV_ROW_ORDER)
    .select(MONITORED_COLS)
    .collect(engine="streaming")
)
SAMPLE_PAYLOAD = {}
for _col in MONITORED_COLS:
    _val = _raw_sample[0, _col]
    SAMPLE_PAYLOAD[_col] = None if _val is None else float(_val)

_score_resp = client.post("/score", json={
    "customer_id": SAMPLE_CUSTOMER_ID,
    "current_statement": SAMPLE_PAYLOAD,
    "previous_state": SAMPLE_PREVIOUS_STATE,
})
assert _score_resp.status_code == 200, f"/score returned {_score_resp.status_code}: {_score_resp.text}"
_api_result = _score_resp.json()
print(f"POST /score     -> {_score_resp.status_code}  state={_api_result['state']}  "
      f"severity_score={_api_result['severity_score']:.6f}  escalated={_api_result['escalated']}")

_score_diff = abs(_api_result["severity_score"] - SAMPLE_EXPECTED_SCORE)
_state_matches = _api_result["state"] == SAMPLE_EXPECTED_STATE
_transition_probs_match = _api_result["transition_probabilities"] == TRANSITION_MATRIX[SAMPLE_PREVIOUS_STATE]
print(f"\nEnd-to-end check: API severity_score ({_api_result['severity_score']:.6f}) vs. directly-computed "
      f"({SAMPLE_EXPECTED_SCORE:.6f}) -- diff {_score_diff:.8f}")
print(f"End-to-end check: API state ({_api_result['state']}) vs. directly-computed ({SAMPLE_EXPECTED_STATE}) -- match: {_state_matches}")
print(f"End-to-end check: API transition_probabilities match TRANSITION_MATRIX[{SAMPLE_PREVIOUS_STATE!r}]: {_transition_probs_match}")

API_SELF_TEST_PASSED = bool(_score_diff < 1e-6 and _state_matches and _transition_probs_match)
if API_SELF_TEST_PASSED:
    print("\n✅ MATCH -- the live API's scoring and transition lookup are verified consistent with direct computation.")
else:
    print("\n❌ MISMATCH -- do not deploy roll_rate_scoring_service.py until this is resolved.")
if not API_SELF_TEST_PASSED:
    raise RuntimeError("Notebook 48's API self-test FAILED -- see ❌ line above. Not safe to proceed.")
print("\n✅ Section 12 complete.")


# =============================================================================
# SECTION 13: API LATENCY BENCHMARK
# =============================================================================
_section("SECTION 13: API Latency Benchmark")

N_API_LATENCY_SAMPLES = 150
_api_latencies_ms = []
_latency_payload = {
    "customer_id": SAMPLE_CUSTOMER_ID, "current_statement": SAMPLE_PAYLOAD, "previous_state": SAMPLE_PREVIOUS_STATE,
}
for _ in range(N_API_LATENCY_SAMPLES):
    _t0 = time.perf_counter()
    _ = client.post("/score", json=_latency_payload)
    _api_latencies_ms.append((time.perf_counter() - _t0) * 1000.0)
_api_latencies_ms = np.array(_api_latencies_ms)
api_latency_summary = {
    "n_samples": N_API_LATENCY_SAMPLES,
    "p50_ms": round(float(np.percentile(_api_latencies_ms, 50)), 3),
    "p95_ms": round(float(np.percentile(_api_latencies_ms, 95)), 3),
    "p99_ms": round(float(np.percentile(_api_latencies_ms, 99)), 3),
    "max_ms": round(float(_api_latencies_ms.max()), 3),
}
print(f"/score latency over {N_API_LATENCY_SAMPLES} real TestClient calls: {api_latency_summary}")
print("\n✅ Section 13 complete.")


# =============================================================================
# SECTION 14: DEPLOYMENT READINESS CHECKLIST
# =============================================================================
_section("SECTION 14: Deployment Readiness Checklist")

deployment_readiness_rows = [
    {"dimension": "Notebook 47 reproduction (zero-randomness pipeline)", "status": "PASS" if _reproduction_matches else "FAIL"},
    {"dimension": "Monotonicity KPI (Notebook 46 target)", "status": "MET" if MEETS_MONOTONICITY_KPI else "NOT MET"},
    {"dimension": "Coherence KPI (Notebook 46 target)", "status": "MET" if MEETS_COHERENCE_KPI else "NOT MET"},
    {"dimension": "Full metrics-suite statistical validation (all checks)", "status": "PASS" if ALL_STAT_CHECKS_PASS else "FAIL"},
    {"dimension": "Deployment policy artifact persisted", "status": "PASS" if deployment_policy_path.exists() else "FAIL"},
    {"dimension": "API self-test (live, generated service)", "status": "PASS" if API_SELF_TEST_PASSED else "FAIL"},
    {"dimension": "API p99 latency < 500ms", "status": "PASS" if api_latency_summary["p99_ms"] < 500 else "FAIL"},
    {"dimension": "Overall recommendation",
     "status": "RECOMMENDED FOR PRODUCTION" if MEETS_KPI and ALL_STAT_CHECKS_PASS else "NOT RECOMMENDED FOR PRODUCTION"},
]
deployment_readiness_df = pd.DataFrame(deployment_readiness_rows)
deployment_readiness_path = RR_DEPLOYMENT_DIR / "deployment_readiness_checklist.csv"
deployment_readiness_df.to_csv(deployment_readiness_path, index=False)
print(deployment_readiness_df.to_string(index=False))
print(f"✅ Saved -> {deployment_readiness_path}")
print("\n✅ Section 14 complete.")


# =============================================================================
# SECTION 15: CHARTS
# =============================================================================
_section("SECTION 15: Charts")


def _style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.3)


fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(_valid_ratios, bins=40, color="#2563eb", alpha=0.75, edgecolor="white")
ax.axvline(SEVERE_TO_LOW_RATIO, color="#16a34a", linewidth=2, label=f"Point estimate ({SEVERE_TO_LOW_RATIO:.2f}x)")
ax.axvline(RATIO_CI_LOWER, color="#dc2626", linestyle="--", label=f"95% CI [{RATIO_CI_LOWER:.2f}, {RATIO_CI_UPPER:.2f}]")
ax.axvline(RATIO_CI_UPPER, color="#dc2626", linestyle="--")
ax.axvline(RR_KPI_TARGETS["min_default_rate_ratio_top_to_bottom_tier"], color="#7c3aed", linestyle=":",
           label=f"KPI target ({RR_KPI_TARGETS['min_default_rate_ratio_top_to_bottom_tier']}x)")
ax.set_xlabel("Severe / Low default-rate ratio (bootstrap resample)")
ax.set_ylabel("Count (of 2,000 bootstrap draws)")
ax.set_title("Bootstrap Distribution -- Monotonicity Ratio (Primary KPI)")
ax.legend(fontsize=8)
_style_axes(ax)
chart1_path = CHARTS_DIR / "notebook_48_bootstrap_ratio_distribution.png"
fig.tight_layout()
fig.savefig(chart1_path, dpi=150)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(_valid_gaps, bins=40, color="#2563eb", alpha=0.75, edgecolor="white")
ax.axvline(P_SEVERE_SEVERE - P_LOW_SEVERE, color="#16a34a", linewidth=2,
           label=f"Point estimate ({P_SEVERE_SEVERE - P_LOW_SEVERE:.3f})")
ax.axvline(COHERENCE_GAP_CI_LOWER, color="#dc2626", linestyle="--",
           label=f"95% CI [{COHERENCE_GAP_CI_LOWER:.3f}, {COHERENCE_GAP_CI_UPPER:.3f}]")
ax.axvline(COHERENCE_GAP_CI_UPPER, color="#dc2626", linestyle="--")
ax.axvline(0.0, color="#64748b", linestyle=":", label="No-effect (0.0)")
ax.set_xlabel("P(Severe->Severe) - P(Low->Severe)  (bootstrap resample)")
ax.set_ylabel("Count (of 2,000 bootstrap draws)")
ax.set_title("Bootstrap Distribution -- Transition Coherence Gap (Second Hard-Gate KPI)")
ax.legend(fontsize=8)
_style_axes(ax)
chart2_path = CHARTS_DIR / "notebook_48_bootstrap_coherence_gap_distribution.png"
fig.tight_layout()
fig.savefig(chart2_path, dpi=150)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5))
_header_labels = STATE_NAMES
_matrix_vals = np.array([[TRANSITION_MATRIX[_i][_j] for _j in STATE_NAMES] for _i in STATE_NAMES])
_im = ax.imshow(_matrix_vals, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(_header_labels)))
ax.set_yticks(range(len(_header_labels)))
ax.set_xticklabels(_header_labels)
ax.set_yticklabels(_header_labels)
ax.set_xlabel("Next state")
ax.set_ylabel("Current state")
ax.set_title("Real Empirical Transition-Probability Matrix (Reproduced)")
for _i in range(len(_header_labels)):
    for _j in range(len(_header_labels)):
        ax.text(_j, _i, f"{_matrix_vals[_i, _j]:.3f}", ha="center", va="center",
                color="white" if _matrix_vals[_i, _j] > 0.5 else "black")
fig.colorbar(_im, ax=ax, label="P(next | current)")
chart3_path = CHARTS_DIR / "notebook_48_transition_matrix_heatmap.png"
fig.tight_layout()
fig.savefig(chart3_path, dpi=150)
plt.show()
plt.close(fig)

print(f"Saved: {chart1_path.name}, {chart2_path.name}, {chart3_path.name}")
print("\n✅ Section 15 complete.")


# =============================================================================
# SECTION 16: WORD REPORT -- Roll_Rate_Validation_Deployment_Report.docx
# =============================================================================
_section("SECTION 16: Word Report -- Roll_Rate_Validation_Deployment_Report.docx")

doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 3, Problem 8: Roll-Rate Modeling -- Validation & Deployment Report")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

doc.add_heading("1. Scope & Technique", level=1)
doc.add_paragraph(
    "A Markov transition-probability matrix over a per-statement delinquency-severity score, reusing "
    "Problem 4's real feature universe and tier names but fitting fresh statement-level weights and "
    "cutpoints (Notebook 47). This notebook independently rebuilds Notebook 47's entire pipeline from "
    "scratch and confirms it reproduces byte-for-byte (zero randomness involved), then bootstraps "
    "confidence intervals on this problem's two real hard-gate KPIs. "
    + ("Both hard gates were met on this real run." if MEETS_KPI else
       "IMPORTANT: at least one real hard gate was NOT met on this real run -- see Section 3 below for "
       "the exact measured values. This report packages the technique for completeness and transparency, "
       "and the recommendation below reflects this honestly.")
)

doc.add_heading("2. Full Statistical Validation Summary", level=1)
doc.add_paragraph(
    "Per the platform's standing metrics-suite directive (effective Problem 6 onward): every metric "
    "below is a real, measured value reproduced independently in this notebook, cross-checked against "
    "Notebook 47's originally-reported numbers (see Section 4's reproduction check)."
)
_t = doc.add_table(rows=1, cols=len(statistical_validation_df.columns))
_t.style = "Light Grid Accent 1"
for _i, _col in enumerate(statistical_validation_df.columns):
    _t.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in statistical_validation_df.iterrows():
    _cells = _t.add_row().cells
    for _i, _col in enumerate(statistical_validation_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("3. Honest Limitation -- Deployment Scope & Assumptions", level=1)
for _k, _v in DEPLOYMENT_LIMITATION.items():
    doc.add_paragraph(f"{_k.replace('_', ' ').title()}: {_v}")

doc.add_heading("4. Deployment Readiness Checklist", level=1)
_t3 = doc.add_table(rows=1, cols=len(deployment_readiness_df.columns))
_t3.style = "Light Grid Accent 1"
for _i, _col in enumerate(deployment_readiness_df.columns):
    _t3.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in deployment_readiness_df.iterrows():
    _cells = _t3.add_row().cells
    for _i, _col in enumerate(deployment_readiness_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("5. API Performance", level=1)
doc.add_paragraph(f"Latency over {api_latency_summary['n_samples']} real TestClient calls to /score: "
                   f"p50={api_latency_summary['p50_ms']}ms, p95={api_latency_summary['p95_ms']}ms, "
                   f"p99={api_latency_summary['p99_ms']}ms, max={api_latency_summary['max_ms']}ms.")

doc.add_heading("6. Charts", level=1)
_chart_entries = [
    (chart1_path, "Bootstrap distribution -- monotonicity ratio (primary KPI)"),
    (chart2_path, "Bootstrap distribution -- transition coherence gap (second hard-gate KPI)"),
    (chart3_path, "Real empirical transition-probability matrix (reproduced)"),
]
for _cp, _cap in _chart_entries:
    doc.add_picture(str(_cp), width=Inches(6.0))
    _p = doc.add_paragraph(_cap)
    _p.alignment = WD_ALIGN_PARAGRAPH.CENTER

report_path = RR_DEPLOYMENT_DIR / "Roll_Rate_Validation_Deployment_Report.docx"
doc.save(report_path)
print(f"✅ Saved -> {report_path}")
print("\n✅ Section 16 complete.")


# =============================================================================
# SECTION 17: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 17: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Notebook 47 reproduction matches (zero-randomness pipeline)", _reproduction_matches)
_all_checks_passed &= _check("Bootstrap ratio CI is well-formed (lower <= point estimate <= upper)",
                              RATIO_CI_LOWER <= SEVERE_TO_LOW_RATIO <= RATIO_CI_UPPER)
_all_checks_passed &= _check("Bootstrap coherence-gap CI is well-formed (lower <= point estimate <= upper)",
                              COHERENCE_GAP_CI_LOWER <= (P_SEVERE_SEVERE - P_LOW_SEVERE) <= COHERENCE_GAP_CI_UPPER)
_all_checks_passed &= _check("Bootstrap AUC CI is well-formed (lower <= point estimate <= upper)",
                              AUC_CI_LOWER <= ESCALATION_METRICS_SUITE["roc_auc_magnitude"] <= AUC_CI_UPPER
                              if ESCALATION_METRICS_SUITE["roc_auc_magnitude"] is not None else True)
_all_checks_passed &= _check("Statistical validation table has 13 rows (every planned test present)",
                              len(statistical_validation_df) == 13)
_all_checks_passed &= _check("Deployment readiness checklist has 8 rows", len(deployment_readiness_df) == 8)
_all_checks_passed &= _check("Generated service compiles (already checked at generation time, re-asserted)",
                              service_py_path.exists() and service_py_path.stat().st_size > 0)
_all_checks_passed &= _check("API self-test passed", API_SELF_TEST_PASSED)
_all_checks_passed &= _check("API p99 latency is a real positive number", api_latency_summary["p99_ms"] > 0)
_all_checks_passed &= _check("recommended_for_production in the persisted policy matches MEETS_KPI and ALL_STAT_CHECKS_PASS",
                              ROLL_RATE_DEPLOYMENT_POLICY["recommended_for_production"] == bool(MEETS_KPI and ALL_STAT_CHECKS_PASS))
for _p in (deployment_policy_path, service_py_path, env_example_path, requirements_api_path,
           statistical_validation_path, deployment_readiness_path, report_path,
           chart1_path, chart2_path, chart3_path):
    _all_checks_passed &= _check(f"{_p.name} exists and is non-empty", _p.exists() and _p.stat().st_size > 0)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 17 complete -- all checks passed.")


# =============================================================================
# SECTION 18: WRITE NOTEBOOK 48 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 18: Write Notebook 48 Summary Artifact")

NB48_SUMMARY = {
    "notebook": "48_roll_rate_modeling_validation_deployment.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "reproduction_matches_notebook_47": _reproduction_matches,
    "meets_monotonicity_kpi": MEETS_MONOTONICITY_KPI,
    "meets_coherence_kpi": MEETS_COHERENCE_KPI,
    "meets_kpi_target": MEETS_KPI,
    "all_stat_checks_pass": ALL_STAT_CHECKS_PASS,
    "recommended_for_production": bool(MEETS_KPI and ALL_STAT_CHECKS_PASS),
    "severe_to_low_default_rate_ratio": SEVERE_TO_LOW_RATIO,
    "bootstrap_ratio_ci": [RATIO_CI_LOWER, RATIO_CI_UPPER],
    "coherence_gap": P_SEVERE_SEVERE - P_LOW_SEVERE,
    "bootstrap_coherence_gap_ci": [COHERENCE_GAP_CI_LOWER, COHERENCE_GAP_CI_UPPER],
    "escalation_roc_auc_magnitude": ESCALATION_METRICS_SUITE["roc_auc_magnitude"],
    "bootstrap_auc_ci": [AUC_CI_LOWER, AUC_CI_UPPER],
    "bootstrap_pr_auc_ci": [PR_AUC_CI_LOWER, PR_AUC_CI_UPPER],
    "split_half_severity_score_psi": SCORE_PSI_SPLIT_HALF,
    "deployment_policy_path": str(deployment_policy_path),
    "service_py_path": str(service_py_path),
    "report_path": str(report_path),
    "statistical_validation_path": str(statistical_validation_path),
    "deployment_readiness_path": str(deployment_readiness_path),
    "api_latency_summary": api_latency_summary,
    "random_seed": RANDOM_SEED,
}
NB48_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_48_summary.json"
with open(NB48_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB48_SUMMARY, f, indent=2)
print(f"Wrote: {NB48_SUMMARY_PATH}")

_section("NOTEBOOK 48 COMPLETE")
print(f"Notebook 47 reproduction verified (zero-randomness pipeline): {_reproduction_matches}")
print(f"Monotonicity KPI    : {MEETS_MONOTONICITY_KPI}  (ratio {SEVERE_TO_LOW_RATIO:.2f}x, "
      f"95% CI [{RATIO_CI_LOWER:.2f}, {RATIO_CI_UPPER:.2f}])")
print(f"Coherence KPI       : {MEETS_COHERENCE_KPI}  (gap {P_SEVERE_SEVERE - P_LOW_SEVERE:.4f}, "
      f"95% CI [{COHERENCE_GAP_CI_LOWER:.4f}, {COHERENCE_GAP_CI_UPPER:.4f}])")
print(f"All statistical checks pass : {ALL_STAT_CHECKS_PASS}")
print(f"Recommended for production  : {NB48_SUMMARY['recommended_for_production']}")
print(f"API self-test       : {'PASSED' if API_SELF_TEST_PASSED else 'FAILED'}")
print(f"Word report         : {report_path}")
print(
    "\nNext: 49_roll_rate_modeling_financial_impact_reporting_packaging.ipynb -- elevated Word/HTML "
    "reporting standard (chart+story narrative pattern, multi-tab interactive dashboard with slicers/"
    "calculator), full synthesis of Notebooks 46-48, honest final recommendation, closes Problem 8 and "
    "Phase 3."
)
